# Machine Learning
## XGBoost Prévention Blessures + KMeans Profilage Joueurs

**Objectif :** Deux modèles ML sur les données SportMetrics :

| Modèle | Objectif | Output |
|---|---|---|
| **KMeans** | Profilage tactique des joueurs | `ml_player_clusters` |
| **XGBoost** | Prévention blessures | `ml_injury_predictions` |

**Features :** Consommées depuis `mart_player` et `mart_physical_condition`  
**Résultats :** Écrits dans `sport_metrics.mart.*` → consommés par Power BI

### Modèle 1: KMeans Clustering
**Problématique :** Identifier les profils tactiques des joueurs pour optimiser le lineup.  
**Features :** Points · Assists · Rebounds · Steals · Blocks · Turnover · Plus_minus  
**k=3 clusters :** Scoreur · Playmaker · Défenseur

In [0]:
%pip install scikit-learn

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# 1. Chargement depuis Delta Lake
# ============================================
logger.info("Chargement des données depuis Delta Lake...")

df = spark.sql("""
    SELECT
        Season,
        player_id,
        player_name,
        MAX(position)                               AS Position,
        MAX(start_position)                         AS Start_position,
        AVG(points)                                 AS Points,
        AVG(assists)                                AS Assists,
        AVG(total_rebounds)                         AS Total_rebounds,
        AVG(steals)                                 AS Steals,
        AVG(blocks)                                 AS Blocks,
        AVG(turnover)                               AS Turnover,
        AVG(player_fault)                           AS Player_fault,
        AVG(plus_minus)                             AS Plus_minus,
        ROUND(AVG(performance_score_match), 2)      AS Performance_score_match,
        ROUND(AVG(performance_score_match_min), 2)  AS Performance_score_match_min,
        ROUND(AVG(minutes_played), 2)               AS minutes_played
    FROM sport_metrics.mart.mart_player
    WHERE minutes_played >= 5 AND Season = '2023-2024'
    GROUP BY Season, player_id, player_name
""").toPandas()

logger.info(f"Données chargées : {len(df)} joueurs")

# ============================================
# 2. Clustering KMeans
# ============================================
features = [
    'Points', 'Assists', 'Total_rebounds', 'Steals',
    'Blocks', 'Turnover', 'Player_fault', 'Plus_minus'
]

X = df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Profil'] = kmeans.fit_predict(X_scaled)

logger.info("Clusters calculés :")
logger.info(df.groupby('Profil')[features].mean())

# ============================================
# 3. Nommage des clusters par Performance Score
# Logique : le performance score moyen par cluster
# détermine la hiérarchie sportive
# Stars > Lieutenants > Rotation/Rookies
# ============================================

cluster_perf = df.groupby('Profil')['Performance_score_match'].mean().sort_values(ascending=False)

print("Performance moyenne par cluster :")
print(cluster_perf)

# Attribution selon le rang de performance
profil_map = {}
labels = ['Stars', 'Lieutenants', 'Rotation']

for rank, cluster_id in enumerate(cluster_perf.index):
    profil_map[cluster_id] = labels[rank]

df['Profil_label'] = df['Profil'].map(profil_map)

print("\nRépartition des clusters :")
print(df.groupby(['Profil', 'Profil_label'])['player_name'].apply(list))

print("\nMoyennes par cluster :")
print(df.groupby('Profil_label')[features + ['Performance_score_match']].mean().round(2))

In [0]:
# ============================================
# 4. Écriture résultats dans Delta Lake
# ============================================
results = df[[
    'Season', 'player_id', 'player_name', 'Position', 'Start_position',
    'Profil', 'Profil_label', 'Performance_score_match',
    'Performance_score_match_min', 'minutes_played'
]].copy()

results['player_id'] = results['player_id'].astype(str)
results['Profil']    = results['Profil'].astype(int)

# Pandas → Spark DataFrame → Delta Lake
sdf = spark.createDataFrame(results)

sdf.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable("sport_metrics.mart.ml_player_clusters")

print(f"ml_player_clusters → {len(results)} joueurs écrits dans Delta Lake")
print(results[['player_name', 'Profil_label', 'Performance_score_match']].to_string())

### Modèle 2: XGBoost Prévention Blessures
**Problématique :** Prédire le risque de blessure avant chaque match.  
**Target :** `injury_risk` (0/1)  
**Seuil :** 0.20 optimisé pour maximiser le Recall (faux négatifs coûteux)  
**Réduction intensité :** 15% si prob ≥ 0.20 · 50% si ≥ 0.50 · 80% si ≥ 0.70

In [0]:
%pip install xgboost scikit-learn

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score,precision_score,recall_score
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# 1. Chargement depuis Delta Lake
# ============================================
logger.info("Chargement des données depuis Delta Lake...")

df = spark.table("sport_metrics.mart.mart_physical_condition").toPandas()

logger.info(f"Données chargées : {len(df)} lignes")

# ============================================
# 2. Préparation des features
# ============================================
features = [
    'fi_before_match', 'fi_avg_7d', 'fi_max_7d', 'training_load_7d',
    'fi_avg_28d', 'training_load_28d', 'focus_level', 'minutes_played', 'position'
]

df = df[df["season"] == '2023-2024']
df = df.dropna(subset=features + ['injury_risk'])

# Encodage de la position
df['position'] = LabelEncoder().fit_transform(df['position'].astype(str))

# Feature engineered : ACWR (Acute Chronic Workload Ratio)
# Ratio charge 7j / charge 28j → indicateur clé prévention blessures
df['ACWR'] = df['fi_avg_7d'] / (df['fi_avg_28d'] + 1e-9)

df = df.dropna(subset=features + ['ACWR', 'injury_risk'])

X = df[features + ['ACWR']]
y = df['injury_risk']

logger.info(f"Features prêtes : {len(df)} lignes · {len(features)+1} features")
logger.info(f"Distribution target : {y.value_counts().to_dict()}")



In [0]:
# ============================================
# 3. Entraînement XGBoost
# Gestion déséquilibre classes via scale_pos_weight
# ============================================
logger.info("Entraînement du modèle XGBoost...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Calcul du poids pour compenser le déséquilibre
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

logger.info(f"scale_pos_weight : {scale_pos_weight:.2f} (neg={neg}, pos={pos})")

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

# Prédictions sur l'ensemble de test
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# seuil plus bas pour mieux détecter les blessures
for seuil in [0.20, 0.30, 0.40, 0.5]:
    y_pred = (y_prob >= seuil).astype(int)
    print(f"\nSeuil = {seuil}")
    print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
    print("Precision blessure :", round(precision_score(y_test, y_pred), 3))
    print("Recall blessure :", round(recall_score(y_test, y_pred), 3))




In [0]:
# ============================================
# 4. Prédictions sur toutes les données
# Seuil 0.20 → maximise le Recall
# Faux négatif = joueur blessé non détecté
# → coût médical/sportif élevé
# ============================================
logger.info("Calcul des prédictions...")

df['injury_probability'] = xgb_model.predict_proba(X[features + ['ACWR']])[:, 1]
df['injury_alert']       = (df['injury_probability'] >= 0.20).astype(int)

def get_reduction(prob):
    if prob >= 0.70:
        return 80    # Danger → repos quasi total
    elif prob >= 0.50:
        return 50    # Risque élevé → entraînement léger
    elif prob >= 0.20:
        return 15    # Risque modéré → vigilance
    else:
        return 0     # OK → entraînement normal

df['training_intensity_reduction'] = df['injury_probability'].apply(get_reduction)

print(f"\n Résumé des prédictions :")
print(f"Total prédictions    : {len(df)}")
print(f"Alertes blessures    : {df['injury_alert'].sum()}")
print(f"Réduction 15%        : {(df['training_intensity_reduction'] == 15).sum()}")
print(f"Réduction 50%        : {(df['training_intensity_reduction'] == 50).sum()}")
print(f"Réduction 80%        : {(df['training_intensity_reduction'] == 80).sum()}")




In [0]:
# ============================================
# 5. Écriture résultats dans Delta Lake
# ============================================
logger.info("Écriture des résultats dans Delta Lake...")

results = df[[
    'player_id', 'session_id', 'season',
    'injury_probability', 'injury_alert',
    'training_intensity_reduction'
]].copy()

results['player_id']          = results['player_id'].astype(str)
results['injury_probability']  = results['injury_probability'].round(4)

sdf = spark.createDataFrame(results)

sdf.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable("sport_metrics.mart.ml_injury_predictions")

print(f"ml_injury_predictions → {len(results)} lignes écrites dans Delta Lake")